# 🚀 Model 6: Complex Hybrid LSTM + CNN + Conformer

**Input:** `(batch, N_BANDS, n_channels)`  
**Feature output:** 256-D penultimate vector → SVM  
Three parallel branches fused + refined by a final Conformer block.

## 0. Mount Google Drive
Required since this notebook reads/writes files under `/content/drive/MyDrive/...`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or '⚠️ No GPU')

Sun Oct  4 13:44:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install scikit-learn openpyxl -q

In [4]:
import os,gc,time,numpy as np,pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers,models,Input,optimizers as opt_module

BAND_NAMES=["Delta","Theta","Alpha","Beta","Gamma"]

def load_dataset(p):
    df = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype(np.float32)
    n_nan = df.isna().any(axis=1).sum()
    if n_nan:
        print(f"    [nan guard] dropping {n_nan} rows with NaN features")
        df = df.dropna().reset_index(drop=True)
    return df

def normalize_tensors(Xtr, Xv, Xte):
    """Fit mean/std on TRAIN split only (per-feature), apply to all splits."""
    mean = Xtr.mean(axis=0, keepdims=True)
    std  = Xtr.std(axis=0, keepdims=True)
    Xtr  = (Xtr - mean) / (std + 1e-8)
    Xv   = (Xv  - mean) / (std + 1e-8)
    Xte  = (Xte - mean) / (std + 1e-8)
    return Xtr, Xv, Xte

def get_band_columns(df): return {b:[c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}
def prepare_tensors_seq(df):
    bc=get_band_columns(df)
    X=np.stack([df[bc[b]].values for b in BAND_NAMES],axis=1)  # (N,5,n_ch)
    return X, df["label"].values.astype(int)
def evaluate_with_svm(Xtr,ytr,Xte,yte,max_train=20000):
    if len(Xtr) > max_train:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(Xtr), max_train, replace=False)
        Xtr, ytr = Xtr[idx], ytr[idx]
    sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
    svm=SVC(kernel="rbf",C=1.0,gamma="scale",class_weight="balanced",random_state=42)
    svm.fit(Xtr,ytr); p=svm.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {"accuracy":round(accuracy_score(yte,p),4),
            "sensitivity":round(sensitivity,4),
            "specificity":round(specificity,4),
            "f1_macro":round(f1_score(yte,p,average="macro"),4),
            "f1_weighted":round(f1_score(yte,p,average="weighted"),4),
            "report":classification_report(yte,p,labels=[0,1],target_names=["No Seizure","Seizure"])}
def save_results(row,path):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    pd.DataFrame([row]).to_csv(path,mode="a",header=not os.path.exists(path),index=False)
    print("Saved:",row)
def compile_model(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                       tf.keras.metrics.Precision(name="precision"),
                       tf.keras.metrics.Recall(name="recall")])
    return m


In [5]:
DATASET_A  = "/content/drive/MyDrive/epilepsy_outputs/dataset_A.csv"
DATASET_B  = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS    = "/content/drive/MyDrive/epilepsy_outputs/results.csv"
OUTPUT_DIR = "/content/drive/MyDrive/epilepsy_outputs"
EPOCHS=50; BATCH_SIZE=64; PATIENCE=8; SEED=42
MODEL_NAME = "complex_hybrid"

## Architecture

```
Branch A (CNN)       : TimeDistributed Conv1D x2 -> GAP per band -> Dense(d_model)
Branch B (LSTM)      : Dense(d_model) -> BiLSTM(64) -> BN -> Dense(d_model)
Branch C (Conformer) : Dense(d_model) -> N × ConformerBlock

Fusion: Concat [A,B,C] -> Dense(d_model) -> ConformerBlock -> GAP
     -> Dense(256, ReLU) ← feature vector -> Dense(1, sigmoid)
```

In [6]:
class FeedForwardModule(layers.Layer):
    def __init__(self,d,ff=4,dr=0.1,**kw):
        super().__init__(**kw)
        self.ln=layers.LayerNormalization(); self.ff1=layers.Dense(d*ff,activation="swish")
        self.dr1=layers.Dropout(dr); self.ff2=layers.Dense(d); self.dr2=layers.Dropout(dr)
    def call(self,x,training=False):
        r=x; x=self.ln(x); x=self.ff1(x); x=self.dr1(x,training=training)
        x=self.ff2(x); x=self.dr2(x,training=training); return 0.5*x+r

class ConvolutionModule(layers.Layer):
    def __init__(self,d,k=3,dr=0.1,**kw):
        super().__init__(**kw)
        self.ln=layers.LayerNormalization(); self.pw1=layers.Conv1D(d*2,1)
        self.dw=layers.DepthwiseConv1D(k,padding="same",use_bias=False)
        self.bn=layers.BatchNormalization(); self.pw2=layers.Conv1D(d,1); self.dr=layers.Dropout(dr)
    def call(self,x,training=False):
        import tensorflow as tf
        r=x; x=self.ln(x); x=self.pw1(x); x1,x2=tf.split(x,2,axis=-1)
        x=x1*tf.nn.sigmoid(x2); x=self.dw(x); x=self.bn(x,training=training)
        x=tf.nn.swish(x); x=self.pw2(x); x=self.dr(x,training=training); return x+r

class ConformerBlock(layers.Layer):
    def __init__(self,d,h=4,k=3,dr=0.1,**kw):
        super().__init__(**kw)
        self.ff1=FeedForwardModule(d,dr=dr); self.ln_sa=layers.LayerNormalization()
        self.mhsa=layers.MultiHeadAttention(num_heads=h,key_dim=d//h,dropout=dr)
        self.dr_sa=layers.Dropout(dr); self.conv=ConvolutionModule(d,k,dr)
        self.ff2=FeedForwardModule(d,dr=dr); self.ln=layers.LayerNormalization()
    def call(self,x,training=False):
        x=self.ff1(x,training=training); r=x; xn=self.ln_sa(x)
        x=self.mhsa(xn,xn,training=training); x=self.dr_sa(x,training=training); x=x+r
        x=self.conv(x,training=training); x=self.ff2(x,training=training); return self.ln(x)

def build_complex_hybrid(n_bands=5,n_channels=18,cnn_filters=[32,64],lstm_units=64,
                          d_model=64,num_heads=4,num_blocks=2,feature_dim=256,dropout=0.2,output_bias=None):
    bias_init = tf.keras.initializers.Constant(output_bias) if output_bias is not None else "zeros"
    inp=Input(shape=(n_bands,n_channels),name="eeg_input")
    # Branch A: CNN
    a=layers.Reshape((n_bands,n_channels,1))(inp)
    for i,f in enumerate(cnn_filters):
        a=layers.TimeDistributed(layers.Conv1D(f,3,padding="same",activation="relu"),name=f"cnn_td{i}")(a)
        a=layers.TimeDistributed(layers.BatchNormalization(),name=f"cnn_bn{i}")(a)
    a=layers.TimeDistributed(layers.GlobalAveragePooling1D(),name="cnn_gap")(a)
    a=layers.Dense(d_model,activation="relu",name="cnn_proj")(a)
    # Branch B: BiLSTM
    b=layers.Dense(d_model,name="lstm_proj_in")(inp)
    b=layers.Bidirectional(layers.LSTM(lstm_units,return_sequences=True,dropout=dropout),name="bilstm")(b)
    b=layers.BatchNormalization()(b); b=layers.Dense(d_model,activation="relu",name="lstm_proj_out")(b)
    # Branch C: Conformer
    c=layers.Dense(d_model,name="conf_proj_in")(inp)
    for i in range(num_blocks):
        c=ConformerBlock(d_model,num_heads,dr=dropout,name=f"conf_branch_{i}")(c)
    # Fusion
    fused=layers.Concatenate(axis=-1,name="fusion")([a,b,c])
    fused=layers.Dense(d_model,activation="relu",name="fusion_proj")(fused)
    fused=layers.Dropout(dropout)(fused)
    fused=ConformerBlock(d_model,num_heads,dr=dropout,name="conf_fusion")(fused)
    fused=layers.GlobalAveragePooling1D()(fused)
    feat=layers.Dense(feature_dim,activation="relu",name="feature_vector")(fused)
    feat=layers.Dropout(dropout)(feat)
    out=layers.Dense(1,activation="sigmoid",name="output",bias_initializer=bias_init)(feat)
    return (models.Model(inp,out,name="ComplexHybrid"),
            models.Model(inp,feat,name="ComplexHybrid_Feat"))


In [7]:
model, feat_ext = build_complex_hybrid()
model.summary()
print("Feature extractor output:", feat_ext.output_shape)


Model: "ComplexHybrid"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ eeg_input           │ (None, 5, 18)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 5, 18, 1)  │          0 │ eeg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_td0             │ (None, 5, 18, 32) │        128 │ reshape[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_bn0             │ (None, 5, 18, 32) │        128 │ cnn_td0[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_td1             │ (None, 5, 18, 64) │      6,208 │ cnn_bn0[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_proj_in        │ (None, 5, 64)     │      1,216 │ eeg_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_bn1             │ (None, 5, 18, 64) │        256 │ cnn_td1[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, 5, 128)    │     66,048 │ lstm_proj_in[0][… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conf_proj_in        │ (None, 5, 64)     │      1,216 │ eeg_input[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_gap             │ (None, 5, 64)     │          0 │ cnn_bn1[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 5, 128)    │        512 │ bilstm[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conf_branch_0       │ (None, 5, 64)     │     96,384 │ conf_proj_in[0][… │
│ (ConformerBlock)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_proj (Dense)    │ (None, 5, 64)     │      4,160 │ cnn_gap[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_proj_out       │ (None, 5, 64)     │      8,256 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conf_branch_1       │ (None, 5, 64)     │     96,384 │ conf_branch_0[0]… │
│ (ConformerBlock)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion              │ (None, 5, 192)    │          0 │ cnn_proj[0][0],   │
│ (Concatenate)       │                   │            │ lstm_proj_out[0]… │
│                     │                   │            │ conf_branch_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_proj (Dense) │ (None, 5, 64)     │     12,352 │ fusion[0][0]      │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 406,529 (1.55 MB)

 Trainable params: 405,697 (1.55 MB)

 Non-trainable params: 832 (3.25 KB)

Feature extractor output: (None, 256)


In [8]:
def run_trial(ds_path, ds_label, opt_name, opt_obj):
    print(f"\n{'='*55}\n  {MODEL_NAME} | Dataset {ds_label} | {opt_name}\n{'='*55}")
    df=load_dataset(ds_path); X,y=prepare_tensors_seq(df)
    n_bands,n_ch=X.shape[1],X.shape[2]
    Xtrv,Xte,ytrv,yte=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
    Xtr,Xv,ytr,yv=train_test_split(Xtrv,ytrv,test_size=0.15/0.85,stratify=ytrv,random_state=SEED)
    Xtr,Xv,Xte=normalize_tensors(Xtr,Xv,Xte)   # train-only normalization
    total=len(ytr); n1=ytr.sum(); cw={0:total/(2*(total-n1)),1:total/(2*n1)}
    # Capped bias: this architecture includes Conformer blocks (LayerNorm-heavy),
    # same mitigation applied to the pure Conformer and LSTM-Conformer notebooks
    # to avoid an AUC-stuck-at-0.5 training collapse.
    output_bias=np.clip(np.log(n1/(total-n1)), -3.0, 0.0)
    ckpt_best=os.path.join(OUTPUT_DIR,"checkpoints",f"{MODEL_NAME}_{ds_label}_{opt_name}_best.keras")
    # FIX: resume must use a checkpoint saved EVERY epoch, not just on AUC
    # improvement -- otherwise epochs after the last AUC peak are silently lost
    # on disconnect (confirmed: AUC peaked at epoch 17, collapsed to ~0.5 for
    # epochs 18-27, so the best-only checkpoint stayed frozen at 17 even though
    # training had reached 28 -- resume then re-did those 10 epochs for nothing).
    ckpt_latest=os.path.join(OUTPUT_DIR,"checkpoints",f"{MODEL_NAME}_{ds_label}_{opt_name}_latest.keras")
    epoch_counter=ckpt_latest+".epoch.txt"  # overwritten (not appended) each epoch -- immune to
                                             # stale/duplicated CSV rows if Drive sync lags on account switch
    os.makedirs(os.path.dirname(ckpt_best),exist_ok=True)

    initial_epoch=0
    if os.path.exists(ckpt_latest) and os.path.exists(epoch_counter):
        initial_epoch=int(open(epoch_counter).read().strip())
        print(f"    [resume] found latest checkpoint at epoch {initial_epoch}, continuing from there")
        full_m=tf.keras.models.load_model(ckpt_latest, compile=False, custom_objects={
            "FeedForwardModule": FeedForwardModule, "ConvolutionModule": ConvolutionModule,
            "ConformerBlock": ConformerBlock})
        full_m=compile_model(full_m,opt_obj)
        feat_m=models.Model(full_m.input, full_m.get_layer("feature_vector").output)
    else:
        full_m,feat_m=build_complex_hybrid(n_bands=n_bands,n_channels=n_ch,output_bias=output_bias)
        full_m=compile_model(full_m,opt_obj)

    def _save_epoch_marker(epoch, logs):
        with open(epoch_counter, "w") as f:
            f.write(str(epoch + 1))  # epoch is 0-indexed; +1 = epochs completed so far

    cbs=[tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=PATIENCE,restore_best_weights=True,verbose=1),
         tf.keras.callbacks.ModelCheckpoint(ckpt_best,monitor="val_auc",mode="max",save_best_only=True,verbose=0),
         tf.keras.callbacks.ModelCheckpoint(ckpt_latest,save_best_only=False,verbose=0),  # every epoch, for resume
         tf.keras.callbacks.LambdaCallback(on_epoch_end=_save_epoch_marker),
         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=4,min_lr=1e-6,verbose=1)]
    t0=time.time()
    hist=full_m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=EPOCHS,initial_epoch=initial_epoch,
                    batch_size=BATCH_SIZE,class_weight=cw,callbacks=cbs,verbose=1)
    elapsed=round((time.time()-t0)/60,2)
    best_auc=max(hist.history.get("val_auc",[0]))
    print(f"  Done {elapsed}min | Best AUC:{best_auc:.4f}")
    f_tr=feat_m.predict(Xtr,batch_size=BATCH_SIZE,verbose=0)
    f_te=feat_m.predict(Xte,batch_size=BATCH_SIZE,verbose=0)
    res=evaluate_with_svm(f_tr,ytr,f_te,yte)
    print(f"  SVM Acc:{res['accuracy']} F1:{res['f1_macro']}\n",res["report"])
    save_results({"model":MODEL_NAME,"dataset":f"Dataset_{ds_label}","optimizer":opt_name,
                  "best_val_auc":round(best_auc,4),**{k:v for k,v in res.items() if k!="report"},
                  "train_time_min":elapsed},RESULTS)
    for f in (ckpt_latest, epoch_counter):
        if os.path.exists(f):
            os.remove(f)  # trial finished -- clear resume state
    del full_m,feat_m; gc.collect(); tf.keras.backend.clear_session()

already_done = set()
if os.path.exists(RESULTS):
    _prev = pd.read_csv(RESULTS)
    _prev_this_model = _prev[_prev["model"] == MODEL_NAME]
    already_done = set(zip(_prev_this_model["dataset"], _prev_this_model["optimizer"]))
    if already_done:
        print(f"[resume] found {len(already_done)} completed trial(s) already in {RESULTS}, skipping those:")
        for d, o in sorted(already_done):
            print(f"    - {d} | {o}")

for ds_label,ds_path in [("A",DATASET_A),("B",DATASET_B)]:
    for opt_name,opt_obj in {"Adam":opt_module.Adam(2e-4, clipnorm=1.0),
                              "Nadam":opt_module.Nadam(2e-4, clipnorm=1.0),
                              "AdamW":opt_module.AdamW(2e-4,weight_decay=1e-4, clipnorm=1.0)}.items():
        if (f"Dataset_{ds_label}", opt_name) in already_done:
            print(f"[resume] skipping {MODEL_NAME} | Dataset {ds_label} | {opt_name} (already done)")
            continue
        run_trial(ds_path,ds_label,opt_name,opt_obj)
print(f"\n✅ {MODEL_NAME} all trials done.")


[resume] found 3 completed trial(s) already in /content/drive/MyDrive/epilepsy_outputs/results.csv, skipping those:
    - Dataset_A | Adam
    - Dataset_A | AdamW
    - Dataset_A | Nadam
[resume] skipping complex_hybrid | Dataset A | Adam (already done)
[resume] skipping complex_hybrid | Dataset A | Nadam (already done)
[resume] skipping complex_hybrid | Dataset A | AdamW (already done)

  complex_hybrid | Dataset B | Adam
    [resume] found latest checkpoint at epoch 15, continuing from there


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'conformer_block', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'conformer_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'conformer_block_2', however the

Epoch 16/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 167s 73ms/step - accuracy: 0.9994 - auc: 0.8601 - loss: 1.9471 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9994 - val_auc: 0.8754 - val_loss: 0.0102 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 17/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 128s 71ms/step - accuracy: 0.9994 - auc: 0.8637 - loss: 1.7764 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9994 - val_auc: 0.9114 - val_loss: 0.0093 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 18/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 127s 70ms/step - accuracy: 0.9994 - auc: 0.8881 - loss: 1.6513 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9994 - val_auc: 0.7742 - val_loss: 0.0058 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 2.0000e-04
Epoch 19/50
1798/1798 ━━━━━━━━━━━━━━━━━━━━ 126s 70ms/step - accuracy: 0.9994 - auc: 0.8685 - loss: 1.5608 - precision: 0.000